### Step 2. 데이터 전처리하기

In [1]:
import os
import pandas as pd

datapath = os.getenv('HOME')+'/aiffel/transformer_chatbot/data/ChatbotData .csv'
data = pd.read_csv(datapath)
data.head()

,Q,A,label
0,12시 땡!,하루가 또 가네요.,0
1,1지망 학교 떨어졌어,위로해 드립니다.,0
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0
4,PPL 심하네,눈살이 찌푸려지죠.,0


- 데이터 개수

In [2]:
len(data)

11823

In [3]:
data.isnull().sum()

Q        0
A        0
label    0
dtype: int64

In [4]:
import re

# 전처리 함수
def preprocess_sentence(sentence):
    # 단어와 구두점(punctuation) 사이의 공백 추가
    sentence = re.sub(r"([?.!,])", r" \1 ", sentence)
    # 연속된 공백을 하나의 공백으로 전환
    sentence = re.sub(r'[" "]+', " ", sentence)
    sentence = sentence.strip()
    return sentence

In [5]:
data['Q'][0]

'12시 땡!'

In [6]:
preprocess_sentence(data['Q'][0])

'12시 땡 !'

In [7]:
data['clean_q'] = data['Q'].apply(lambda x : preprocess_sentence(x))
data['clean_a'] = data['A'].apply(lambda x : preprocess_sentence(x))

In [8]:
data.head()

,Q,A,label,clean_q,clean_a
0,12시 땡!,하루가 또 가네요.,0,12시 땡 !,하루가 또 가네요 .
1,1지망 학교 떨어졌어,위로해 드립니다.,0,1지망 학교 떨어졌어,위로해 드립니다 .
2,3박4일 놀러가고 싶다,여행은 언제나 좋죠.,0,3박4일 놀러가고 싶다,여행은 언제나 좋죠 .
3,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠.,0,3박4일 정도 놀러가고 싶다,여행은 언제나 좋죠 .
4,PPL 심하네,눈살이 찌푸려지죠.,0,PPL 심하네,눈살이 찌푸려지죠 .


In [9]:
questions = data['clean_q']
answers = data['clean_a']
questions, answers

(0                         12시 땡 !
 1                     1지망 학교 떨어졌어
 2                    3박4일 놀러가고 싶다
 3                 3박4일 정도 놀러가고 싶다
 4                         PPL 심하네
                    ...           
 11818             훔쳐보는 것도 눈치 보임 .
 11819             훔쳐보는 것도 눈치 보임 .
 11820                흑기사 해주는 짝남 .
 11821    힘든 연애 좋은 연애라는게 무슨 차이일까 ?
 11822                  힘들어서 결혼할까봐
 Name: clean_q, Length: 11823, dtype: object,
 0                      하루가 또 가네요 .
 1                       위로해 드립니다 .
 2                     여행은 언제나 좋죠 .
 3                     여행은 언제나 좋죠 .
 4                      눈살이 찌푸려지죠 .
                    ...            
 11818          티가 나니까 눈치가 보이는 거죠 !
 11819               훔쳐보는 거 티나나봐요 .
 11820                      설렜겠어요 .
 11821    잘 헤어질 수 있는 사이 여부인 거 같아요 .
 11822          도피성 결혼은 하지 않길 바라요 .
 Name: clean_a, Length: 11823, dtype: object)

### Step 3. SubwordTextEncoder 사용하기 (단어장 만들기)

In [10]:
import tensorflow_datasets as tfds

tokenizer = tfds.deprecated.text.SubwordTextEncoder.build_from_corpus(
    questions + answers, target_vocab_size=2**13)

In [11]:
# 시작 토큰과 종료 토큰에 대한 정수 부여.
START_TOKEN, END_TOKEN = [tokenizer.vocab_size], [tokenizer.vocab_size + 1]

# 시작 토큰과 종료 토큰을 고려하여 단어 집합의 크기를 + 2
VOCAB_SIZE = tokenizer.vocab_size + 2

In [12]:
print('START_TOKEN의 번호 :' ,[tokenizer.vocab_size])
print('END_TOKEN의 번호 :' ,[tokenizer.vocab_size + 1])
print('단어장의 크기 :',VOCAB_SIZE)

START_TOKEN의 번호 : [8364]
END_TOKEN의 번호 : [8365]
단어장의 크기 : 8366


#### 정수 인코딩 & 패딩

In [13]:
# 임의의 22번째 샘플에 대해서 정수 인코딩 작업을 수행.
# 각 토큰을 고유한 정수로 변환
print('정수 인코딩 후의 21번째 질문 샘플: {}'.format(tokenizer.encode(questions[21])))
print('정수 인코딩 후의 21번째 답변 샘플: {}'.format(tokenizer.encode(answers[21])))

정수 인코딩 후의 21번째 질문 샘플: [5829, 605, 2500, 4174]
정수 인코딩 후의 21번째 답변 샘플: [2685, 7669, 8, 6378, 95, 1]


In [14]:
# encode() : 텍스트 시퀀스 --> 정수 시퀀스
tokenized_string = tokenizer.encode(questions[21])
print ('정수 인코딩 후의 문장 {}'.format(tokenized_string))

# decode() : 정수 시퀀스 --> 텍스트 시퀀스
original_string = tokenizer.decode(tokenized_string)
print ('기존 문장: {}'.format(original_string))

# encode() : 텍스트 시퀀스 --> 정수 시퀀스
tokenized_string = tokenizer.encode(answers[21])
print ('정수 인코딩 후의 문장 {}'.format(tokenized_string))

# decode() : 정수 시퀀스 --> 텍스트 시퀀스
original_string = tokenizer.decode(tokenized_string)
print ('기존 문장: {}'.format(original_string))

정수 인코딩 후의 문장 [5829, 605, 2500, 4174]
기존 문장: 가스비 장난 아님
정수 인코딩 후의 문장 [2685, 7669, 8, 6378, 95, 1]
기존 문장: 다음 달에는 더 절약해봐요 .


In [15]:
questions[0]

'12시 땡 !'

In [16]:
import tensorflow as tf

# 최대 길이를 40으로 정의
MAX_LENGTH = 40

def tokenize_and_filter(inputs, outputs):
    tokenized_inputs, tokenized_outputs = [], []

    for (sentence1, sentence2) in zip(inputs, outputs):
        # encode(토큰화 + 정수 인코딩), 시작 토큰과 종료 토큰 추가
        sentence1 = START_TOKEN + tokenizer.encode(sentence1) + END_TOKEN
        sentence2 = START_TOKEN + tokenizer.encode(sentence2) + END_TOKEN

        tokenized_inputs.append(sentence1)
        tokenized_outputs.append(sentence2)

    # 패딩
    tokenized_inputs = tf.keras.preprocessing.sequence.pad_sequences(
        tokenized_inputs, maxlen=MAX_LENGTH, padding='post')
    tokenized_outputs = tf.keras.preprocessing.sequence.pad_sequences(
        tokenized_outputs, maxlen=MAX_LENGTH, padding='post')

    return tokenized_inputs, tokenized_outputs

In [17]:
questions, answers = tokenize_and_filter(questions, answers)
print('단어장의 크기 :',(VOCAB_SIZE))
print('필터링 후의 질문 샘플 개수: {}'.format(len(questions)))
print('필터링 후의 답변 샘플 개수: {}'.format(len(answers)))
questions.shape, answers.shape

단어장의 크기 : 8366
필터링 후의 질문 샘플 개수: 11823
필터링 후의 답변 샘플 개수: 11823


((11823, 40), (11823, 40))

#### Teacher forcing 사용하기

In [18]:
# 텐서플로우 dataset을 이용하여 셔플(shuffle)을 수행하되, 배치 크기로 데이터를 묶는다.
# 또한 이 과정에서 교사 강요(teacher forcing)을 사용하기 위해서 디코더의 입력과 실제값 시퀀스를 구성한다.
BATCH_SIZE = 64
BUFFER_SIZE = 20000

# 디코더는 이전의 target을 다음의 input으로 사용
# 이에 따라 outputs에서는 START_TOKEN을 제거
dataset = tf.data.Dataset.from_tensor_slices((
    {
        'inputs': questions,
        'dec_inputs': answers[:, :-1] # 마지막 패딩 토큰 제거
    },
    {
        'outputs': answers[:, 1:]  # 시작 토큰이 제거
    },
))

dataset = dataset.cache()
dataset = dataset.shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE)
dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)

### Step 4. 모델 구성하기

- PositionalEncoding

In [19]:
# 포지셔널 인코딩 레이어
class PositionalEncoding(tf.keras.layers.Layer):

    def __init__(self, position, d_model):
        super(PositionalEncoding, self).__init__()
        self.pos_encoding = self.positional_encoding(position, d_model)

    def get_angles(self, position, i, d_model):
        angles = 1 / tf.pow(10000, (2 * (i // 2)) / tf.cast(d_model, tf.float32))
        return position * angles

    def positional_encoding(self, position, d_model):
        # 각도 배열 생성
        angle_rads = self.get_angles(
            position=tf.range(position, dtype=tf.float32)[:, tf.newaxis],
            i=tf.range(d_model, dtype=tf.float32)[tf.newaxis, :],
            d_model=d_model)

        # 배열의 짝수 인덱스에는 sin 함수 적용
        sines = tf.math.sin(angle_rads[:, 0::2])
        # 배열의 홀수 인덱스에는 cosine 함수 적용
        cosines = tf.math.cos(angle_rads[:, 1::2])

        # sin과 cosine이 교차되도록 재배열
        pos_encoding = tf.stack([sines, cosines], axis=0)
        pos_encoding = tf.transpose(pos_encoding,[1, 2, 0])
        pos_encoding = tf.reshape(pos_encoding, [position, d_model])

        # angle_rads = np.zeros(angle_rads.shape)
        # angle_rads[:, 0::2] = sines
        # angle_rads[:, 1::2] = cosines
        # pos_encoding = tf.constant(angle_rads)

        pos_encoding = pos_encoding[tf.newaxis, ...]

        return tf.cast(pos_encoding, tf.float32)

    def call(self, inputs):
        return inputs + self.pos_encoding[:, :tf.shape(inputs)[1], :]

- MultiHeadAttention

In [20]:
# 스케일드 닷 프로덕트 어텐션 함수
def scaled_dot_product_attention(query, key, value, mask):
    # 어텐션 가중치는 Q와 K의 닷 프로덕트
    matmul_qk = tf.matmul(query, key, transpose_b=True)

    # 가중치를 정규화
    depth = tf.cast(tf.shape(key)[-1], tf.float32)
    logits = matmul_qk / tf.math.sqrt(depth)

    # 패딩에 마스크 추가
    if mask is not None:
        logits += (mask * -1e9)

    # softmax적용
    attention_weights = tf.nn.softmax(logits, axis=-1)

    # 최종 어텐션은 가중치와 V의 닷 프로덕트
    output = tf.matmul(attention_weights, value)
    return output

class MultiHeadAttention(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, name="multi_head_attention"):
        super(MultiHeadAttention, self).__init__(name=name)
        self.num_heads = num_heads
        self.d_model = d_model

        assert d_model % self.num_heads == 0

        self.depth = d_model // self.num_heads

        self.query_dense = tf.keras.layers.Dense(units=d_model)
        self.key_dense = tf.keras.layers.Dense(units=d_model)
        self.value_dense = tf.keras.layers.Dense(units=d_model)

        self.dense = tf.keras.layers.Dense(units=d_model)

    def split_heads(self, inputs, batch_size):
        inputs = tf.reshape(
            inputs, shape=(batch_size, -1, self.num_heads, self.depth))
        return tf.transpose(inputs, perm=[0, 2, 1, 3])

    def call(self, inputs):
        query, key, value, mask = inputs['query'], inputs['key'], inputs['value'], inputs['mask']
        batch_size = tf.shape(query)[0]

        # Q, K, V에 각각 Dense를 적용합니다
        # 1. WQ, WK, WV에 해당하는 밀집층 지나기
        # q : (batch_size, query의 문장 길이, d_model)
        # k : (batch_size, key의 문장 길이, d_model)
        # v : (batch_size, value의 문장 길이, d_model)
        # 참고) 인코더(k, v)-디코더(q) 어텐션에서는 query 길이와 key, value의 길이는 다를 수 있다.
        query = self.query_dense(query)
        key = self.key_dense(key)
        value = self.value_dense(value)

        # 병렬 연산을 위한 머리를 여러 개 만듭니다
        # 2. 헤드 나누기
        # q : (batch_size, num_heads, query의 문장 길이, d_model/num_heads)
        # k : (batch_size, num_heads, key의 문장 길이, d_model/num_heads)
        # v : (batch_size, num_heads, value의 문장 길이, d_model/num_heads)
        query = self.split_heads(query, batch_size)
        key = self.split_heads(key, batch_size)
        value = self.split_heads(value, batch_size)

        # 스케일드 닷 프로덕트 어텐션 함수
        scaled_attention = scaled_dot_product_attention(query, key, value, mask)

        scaled_attention = tf.transpose(scaled_attention, perm=[0, 2, 1, 3])

        # 어텐션 연산 후에 각 결과를 다시 연결(concatenate)합니다
        concat_attention = tf.reshape(scaled_attention,
                                      (batch_size, -1, self.d_model))

        # 최종 결과에도 Dense를 한 번 더 적용합니다
        outputs = self.dense(concat_attention)

        return outputs

- create_padding_mask & create_look_ahead_mask

In [21]:
def create_padding_mask(x):
    mask = tf.cast(tf.math.equal(x, 0), tf.float32)
    # (batch_size, 1, 1, sequence length)
    return mask[:, tf.newaxis, tf.newaxis, :]

def create_look_ahead_mask(x):
    seq_len = tf.shape(x)[1]
    # 룩어헤드 마스크
    look_ahead_mask = 1 - tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)
    # 패딩 마스크
    padding_mask = create_padding_mask(x)
    # 마스크 두 개 합치기
    return tf.maximum(look_ahead_mask, padding_mask)

In [22]:
class TransformerDecoderLayer(tf.keras.layers.Layer):
    def __init__(self, d_model, num_heads, dff, rate=0.1):
        super(TransformerDecoderLayer, self).__init__()
        self.mha = MultiHeadAttention(d_model, num_heads)
        self.ffn = tf.keras.Sequential([
            tf.keras.layers.Dense(dff, activation='relu'),
            tf.keras.layers.Dense(d_model)
        ])
        
        self.layernorm1 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        self.layernorm2 = tf.keras.layers.LayerNormalization(epsilon=1e-6)
        
        self.dropout1 = tf.keras.layers.Dropout(rate)
        self.dropout2 = tf.keras.layers.Dropout(rate)

    def call(self, x, training, mask):  
        # ✅ 올바른 입력 딕셔너리 형태로 MultiHeadAttention에 전달
        attn_output = self.mha({
            'query': x,
            'key': x,
            'value': x,
            'mask': mask
        })  

        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.layernorm1(x + attn_output)
        
        ffn_output = self.ffn(out1)
        ffn_output = self.dropout2(ffn_output, training=training)
        out2 = self.layernorm2(out1 + ffn_output)
        return out2


In [23]:
class GPT1(tf.keras.Model):
    def __init__(self, num_layers, d_model, num_heads, dff, vocab_size, max_pos, rate=0.1):
        super(GPT1, self).__init__()
        self.embedding = tf.keras.layers.Embedding(vocab_size, d_model)
        self.pos_encoding = tf.keras.layers.Embedding(max_pos, d_model)
        self.dec_layers = [TransformerDecoderLayer(d_model, num_heads, dff, rate) for _ in range(num_layers)]
        self.dropout = tf.keras.layers.Dropout(rate)
        self.final_layer = tf.keras.layers.Dense(vocab_size)

    def call(self, x, training, mask):
        seq_len = tf.shape(x)[1]
        pos = tf.range(0, seq_len)
        pos = tf.expand_dims(pos, 0)
        x = self.embedding(x) + self.pos_encoding(pos)
        x = self.dropout(x, training=training)
        
        for dec_layer in self.dec_layers:
            x = dec_layer(x, training, mask)
        
        return self.final_layer(x)


In [24]:
BATCH_SIZE = 64
BUFFER_SIZE = 20000

dataset = tf.data.Dataset.from_tensor_slices((
    {
        'inputs': questions,
        'dec_inputs': answers[:, :-1]  # 마지막 패딩 토큰 제거
    },
    {
        'outputs': answers[:, 1:]  # 시작 토큰 제거
    },
))

dataset = dataset.cache()
dataset = dataset.shuffle(BUFFER_SIZE)
dataset = dataset.batch(BATCH_SIZE)
dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)


In [25]:
class GPT1Trainer:
    def __init__(self, model, learning_rate=5e-5):
        self.model = model
        self.loss_object = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
        self.optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)

    def loss_function(self, real, pred):
        mask = tf.math.logical_not(tf.math.equal(real, 0))
        loss_ = self.loss_object(real, pred)
        mask = tf.cast(mask, dtype=loss_.dtype)
        loss_ *= mask
        return tf.reduce_mean(loss_)

    @tf.function
    def train_step(self, inp, tar):
        mask = create_padding_mask(inp)  # ✅ 패딩 마스크 생성

        with tf.GradientTape() as tape:
            predictions = self.model(inp, training=True, mask=mask)

            # ✅ `tar`을 predictions과 동일한 길이로 패딩 적용
            tar_shape = tf.shape(predictions)[1]  # `predictions`의 시퀀스 길이 가져오기
            tar = tf.pad(tar, [[0, 0], [0, tar_shape - tf.shape(tar)[1]]])  # ✅ TensorFlow의 `tf.pad()` 사용

            loss = self.loss_function(tar, predictions)

        gradients = tape.gradient(loss, self.model.trainable_variables)
        self.optimizer.apply_gradients(zip(gradients, self.model.trainable_variables))
        return loss












    def train(self, dataset, epochs=3):
        for epoch in range(epochs):
            total_loss = 0
            for batch in dataset:
                inp = batch[0]['inputs']  # ✅ 명확하게 inputs 추출
                tar = batch[1]['outputs']  # ✅ 명확하게 outputs 추출

                batch_loss = self.train_step(inp, tar)
                total_loss += batch_loss

            print(f'Epoch {epoch+1}, Loss: {total_loss.numpy():.4f}')




In [ ]:
# 속도 최적화된 GPT-1 모델 하이퍼파라미터
NUM_LAYERS = 6 
D_MODEL = 768  
NUM_HEADS = 12 
UNITS = 1024
VOCAB_SIZE = tokenizer.vocab_size + 2
MAX_POS = MAX_LENGTH

# GPT-1 모델 생성
gpt1 = GPT1(NUM_LAYERS, D_MODEL, NUM_HEADS, UNITS, VOCAB_SIZE, MAX_POS)
# Trainer 초기화
trainer = GPT1Trainer(gpt1)
# 학습 시작
trainer.train(dataset, epochs=12)  # 원하는 에포크 수 조절 가능
gpt1.summary()


Epoch 1, Loss: 53.2860
Epoch 2, Loss: 37.8018
Epoch 3, Loss: 36.5834
Epoch 4, Loss: 35.6941
Epoch 5, Loss: 34.9029
Epoch 6, Loss: 34.0996
Epoch 7, Loss: 33.3730
Epoch 8, Loss: 32.4274
Epoch 9, Loss: 31.4869
Epoch 10, Loss: 30.6745
Epoch 11, Loss: 29.8762


In [ ]:
def gpt1_inference(sentence, model):
    """
    GPT-1 모델을 사용하여 문장을 생성하는 함수 (기존 방식 유지)
    
    Args:
        sentence (str): 입력 문장
        model (tf.keras.Model): 훈련된 GPT-1 모델

    Returns:
        tf.Tensor: 생성된 문장의 정수 시퀀스
    """
    sentence = preprocess_sentence(sentence)  # ✅ 전처리 적용
    sentence = tf.expand_dims(START_TOKEN + tokenizer.encode(sentence), axis=0)  # ✅ 토큰화 및 배치 차원 추가

    for _ in range(MAX_LENGTH):
        mask = create_padding_mask(sentence)  # ✅ 패딩 마스크 생성
        predictions = model(sentence, training=False, mask=mask)  # ✅ GPT-1 예측
        predictions = predictions[:, -1:, :]  # ✅ 마지막 단어 확률만 사용

        predicted_id = tf.cast(tf.argmax(predictions, axis=-1), tf.int32)  # ✅ 가장 확률 높은 단어 선택

        if tf.equal(predicted_id, END_TOKEN[0]):  # ✅ 종료 토큰이면 중단
            break

        sentence = tf.concat([sentence, predicted_id], axis=-1)  # ✅ 예측된 단어 추가

    return tf.squeeze(sentence, axis=0)  # ✅ 최종 예측 결과 반환


def sentence_generation(sentence, model):
    """
    입력 문장에 대해 GPT-1 모델이 생성한 문장을 반환

    Args:
        sentence (str): 입력 문장
        model (tf.keras.Model): 훈련된 GPT-1 모델

    Returns:
        str: 생성된 문장
    """
    prediction = gpt1_inference(sentence, model)  # ✅ GPT-1 인퍼런스 수행
    predicted_sentence = tokenizer.decode([i for i in prediction if i < tokenizer.vocab_size])  # ✅ 디코딩

    print(f'입력: {sentence}')
    print(f'출력: {predicted_sentence}\n')

    return predicted_sentence


# 예제 실행
sentence_generation('아 심심해', gpt1)


In [ ]:
sentence_generation('어 너무 싫어', gpt1)

In [ ]:
# 테스트 실행
test_questions = [
    '안녕하세요~~~',
    '아 퇴근 시간 언제야?',
    '왤케 시간이 안 가냐 ㅠ',
    '집에 가고 싶다',
    '몇 시에 만날래?',
    '근황 토크 좀 하자',
    '거기 개쩌는데잖아',
    '나는 헬스를 다니니까',
    '니 이제 인났나?',
    '뭐고?',
    '잤나?'
]

for test in test_questions:
    sentence_generation(test, gpt1)
